In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import kagglehub

import sqlite3
from typing import Union, List
import itertools
from itertools import combinations
import networkx as nx

import json
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Openai") 

client = OpenAI(api_key=api_key)

In [ ]:
import os

In [ ]:
if __name__ == "__main__":
    # test calls, prints, etc.
    import pickle

    # Open the file in binary read mode ('rb')
    with open('/kaggle/input/notebooks/mehulkumar99/preprocessing/spider_schema_lookup.pkl', 'rb') as file:
        schema_lookup = pickle.load(file)
    
    print(schema_lookup['concert_singer']['embeddings'].shape)
    print(schema_lookup['concert_singer']['schema'])
    print(schema_lookup['concert_singer'].keys())

In [ ]:

if __name__ == "__main__":
    desc = f"a column about singer"
    
    response = client.embeddings.create(
    model="text-embedding-3-small",
    input= desc
    )
    
    query_vec = response.data[0].embedding   # returns a list of floats
    print(len(query_vec))

In [ ]:
from sklearn.preprocessing import normalize
def search_column(query, col_dicts, embeddings, topk=5):
    """
    query: list of str — semantic descriptions of needed columns
    col_dicts: list of dicts for this DB (from schema_lookup)
    embeddings: np.array (n_cols, 384) for this DB
    model: SentenceTransformer instance
    topk: number of results per query term
    
    Returns: dict {query_term: [col_dict, ...]} or list if single query
    """

    if not query:
        return 'Nothing was sent to search'
    
    results = {}
    
    for q in query:
        q_norm = q.strip().lower()
        
        # Match their query transformation
        desc = f"a column about {q_norm}."
        
        # Encode query
        response = client.embeddings.create(
        model="text-embedding-3-small",
        input= desc
        )
        query_vec = response.data[0].embedding   # returns a list of floats
        query_vec_norm = query_vec / np.linalg.norm(query_vec)

        
        # Cosine similarity
        scores = embeddings @ query_vec_norm  # (n_cols,)
        # print(score.shape)
        
        # Exact name match override — force to top
        for i, col in enumerate(col_dicts):
            if q_norm == col['column_name'].lower():
                scores[i] = 1.0  # max possible cosine sim
        
        # Rank and take top-k
        top_indices = scores.argsort()[::-1][:topk]
        
        # Build result list
        format_results = []
        for idx in top_indices:
            col = col_dicts[idx]
            
            # Truncate statistics
            stats = col.get('statistics', 'N/A')
            if len(str(stats)) > 200:
                stats = str(stats)[:200] + f"...(Omit {len(str(stats))-200} chars)"
            
            format_results.append({
                'column': col['column_name'],
                'format': col['column_type'],
                'table': col['table_name'],
                'statistics': stats
            })
        
        results[q] = format_results
    
    # Single query → return list directly (match their behavior)
    if len(results) == 1:
        return results[query[0]]
    
    return results

In [ ]:

if __name__ == "__main__":
    db = schema_lookup['concert_singer']
    result = search_column(
        query=['song name', 'singer age', 'song release year'],
        col_dicts=db['col_dicts'],
        embeddings=db['embeddings'],
        topk=3
    )
    print(result)

In [ ]:
def search_value(query, db_id, col_dicts, db_base_path, table = None, column = None):

    """
    query: str — value to search for e.g. "Soccer activity"
    db_id: str — which DB to search in
    col_dicts: list of dicts from schema_lookup
    db_base_path: path to database/ folder
    table: optional — narrow search to this table
    column: optional — narrow search to this column
    
    Returns: list of {'contents': ..., 'table': ..., 'column': ...}
    """

    # Checking if the table and column passed by the model is present in the db, or it hallucinated
    
    table_presence = False
    column_presence = False
    
    if table:
        for col_dic in col_dicts:
            if col_dic['table_name'] == table:
                table_presence = True 
    if column:   
        for col_dic in col_dicts:
            if col_dic['column_name'] == column:
                column_presence = True
    if table and not table_presence:
        return f'{table} named table not present in schema'
    if column and not column_presence:
        return f'{column} named column not present in schema'

    

    # list of stop words to not check since we tokenize and check it

    STOP_WORDS = {'the', 'a', 'an', 'of', 'in', 'at', 'by', 'for',
                  'with', 'about', 'activity', 'area', 'type', 'name'}

    # Connect to DB
    db_path = os.path.join(db_base_path, db_id, f"{db_id}.sqlite")
    conn = sqlite3.connect(db_path)
    conn.text_factory = lambda b: b.decode(errors='ignore')
    cursor = conn.cursor()

    # Tokenize
    tokens = query.lower().split()
    tokens = [t for t in tokens if t not in STOP_WORDS]
    if not tokens:
        tokens = [query]  # fallback if all words were stop words

    
    list_of_contents = []
    search_list = []
    seen = set() # for deduplication.  Same (contents, table, column) can appear multiple times if multiple tokens match the same cell
    

    if table and column:   
        search_list.append((table,column))
        

    elif table and not column:
        for col_dic in col_dicts:
            if col_dic['table_name'] == table:
                search_list.append((table, col_dic['column_name']))
                

    elif not table and not column:
        for col_dic in col_dicts:               
            if col_dic['column_type'] in ('text', 'others'):
                search_list.append((col_dic['table_name'], col_dic['column_name']))

    for token in tokens:
        for search_window in search_list:
            t = search_window[0]
            c = search_window[1]

            try:
                cursor.execute(
                    f"SELECT DISTINCT `{c}` FROM `{t}` "
                    f"WHERE `{c}` LIKE ? AND `{c}` IS NOT NULL",
                    (f'%{token}%',)
                )
                rows = cursor.fetchall()
                for row in rows:
                    
                    content = row[0]
                    key = (str(content), t, c)
                    if key not in seen:
                        seen.add(key)
                        
                        temp = {}
                        temp['contents'] = content
                        temp['table'] = t
                        temp['column'] = c
                        list_of_contents.append(temp)
            except Exception as e:
                print(f"[search_value] query failed on {t}.{c}: {e}")
                continue

    conn.close()

    if not list_of_contents: 
        return "No matching values found."
        
    return list_of_contents

In [ ]:
if __name__ == "__main__":
    db_id = 'activity_1'
    col_dicts = schema_lookup[db_id]['col_dicts']
    query = 'soccer activity'
    db_base_path = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'
    results = search_value(query, db_id, col_dicts, db_base_path, table = None, column = None)
    print(results)

In [ ]:
if __name__ == "__main__":
    db_id = 'activity_1'
    col_dicts = schema_lookup[db_id]['col_dicts']
    query = 'soccer activity'
    db_base_path = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'
    results = search_value(query, db_id, col_dicts, db_base_path, table = 'Activity', column = 'activity')
    print(results)

In [ ]:
if __name__ == "__main__":
    db = schema_lookup['concert_singer']
    G = db['graph']
    
    print("Nodes:", G.number_of_nodes())  # should be 21
    print("Edges:", G.number_of_edges())
    
    # Check FK edges specifically
    print("\nFK edges:")
    for u, v in G.edges():
        # FK edges cross tables
        if u.split('.')[0] != v.split('.')[0]:
            print(f"  {u} <-> {v}")

In [ ]:
def FindShortestPath(db_id, schema_lookup, start, end):

    if not start or not end:
        return "start or end list is empty"


    G = schema_lookup[db_id]['graph']

    pairs = list(itertools.product(start, end)) # [('Student.Fname', 'Activity.activity_name'), ('Faculty.Fname', 'Activity.activity_name')] 
    
    list_of_paths = []
    for pair in pairs:

        source = pair[0]
        target = pair[1]

        try:
            path = nx.shortest_path(G, source, target)
            
        except nx.NetworkXNoPath:
            
            list_of_paths.append((source, target, "no path found"))
            continue
            
        except nx.NodeNotFound:
            
            missing = []
            if source not in G:
                missing.append(source)
            if target not in G:
                missing.append(target)
            msg = f"{' and '.join(missing)} not found in schema"
            list_of_paths.append((source, target, msg))
            continue
            
        temp = path[0]

        for i in range(1, len(path)):
            
            curr_table = path[i].split('.')[0]
            prev_table = path[i-1].split('.')[0]

            if curr_table == prev_table:
                temp+= f' <-> {path[i]}'
            else:
                temp+= f' = {path[i]}'
            
        list_of_paths.append((source, target, temp))

    return list_of_paths

In [ ]:

if __name__ == "__main__":
    list_of_paths = FindShortestPath(db_id = 'activity_1', schema_lookup= schema_lookup,
                        start=["Student.Fname", "Faculty.name"], 
                        end=["Activity.activity_name"],
                        )
    print(list_of_paths)

In [ ]:
def ExecuteSQL(sql: str, db_id: str, db_base_path: str):

    # Connect to DB
    db_path = os.path.join(db_base_path, db_id, f"{db_id}.sqlite")
    conn = sqlite3.connect(db_path)
    conn.text_factory = lambda b: b.decode(errors='ignore')
    cursor = conn.cursor()
    
    try:
        cursor.execute(sql
        )
        rows = cursor.fetchall()
        return ('Successful',rows)

    except Exception as e:
        error_msg = str(e)
        return ( 'Unsuccessful',error_msg)
        
    finally:
        conn.close() # always executes regardless of return or exception

In [ ]:
if __name__ == "__main__":
    result = ExecuteSQL(
        "SELECT Song_Name, Song_release_year FROM singer ORDER BY Age LIMIT 1",
        db_id="concert_singer",
        db_base_path=db_base_path
    )
    print(result)